In [ ]:
from pathlib import Path

import geopandas as gpd
import rasterio


PROJECT_ROOT = Path.cwd().parents[0]

PARCELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "parcels"
    / "Parcelas_Reto_AGC_CONJUNTO.shp"
)

ELEVATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "topography"
    / "Elevacion_INEGI_CEM4_120m.tif"
)

# Cargar parcelas
parcels = gpd.read_file(PARCELS_PATH)

# Transformar SOLO en memoria al CRS del raster
parcels_6372 = parcels.to_crs("EPSG:6372")

# Leer información espacial del raster
with rasterio.open(ELEVATION_PATH) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds

print("CRS original de parcelas:", parcels.crs)
print("CRS transformado:", parcels_6372.crs)
print("CRS del raster:", raster_crs)

print("\nBounds de parcelas transformadas:")
print(parcels_6372.total_bounds)

print("\nBounds del raster:")
print(raster_bounds)

CRS original de parcelas: EPSG:4326
CRS transformado: EPSG:6372
CRS del raster: EPSG:6372

Bounds de parcelas transformadas:
[2854345.49852606  840319.99534535 2896954.08286448  891715.79322106]

Bounds del raster:
BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)


In [ ]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\.venv\Scripts\python.exe
3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]


In [ ]:
# verificamos versiones de estas dos libs
import geopandas as gpd
import rasterio

print("GeoPandas:", gpd.__version__)
print("Rasterio:", rasterio.__version__)

GeoPandas: 1.1.3
Rasterio: 1.5.1


In [ ]:
# Verificar qué parcelas intersectan espacialmente
# con la extensión del raster de elevación

from shapely.geometry import box

raster_polygon = box(
    raster_bounds.left,
    raster_bounds.bottom,
    raster_bounds.right,
    raster_bounds.top,
)

intersects_raster = parcels_6372.geometry.intersects(raster_polygon)

print("Total de parcelas:", len(parcels_6372))
print("Parcelas que intersectan el raster:", intersects_raster.sum())
print("Parcelas fuera del raster:", (~intersects_raster).sum())

Total de parcelas: 197
Parcelas que intersectan el raster: 197
Parcelas fuera del raster: 0


In [ ]:
#cobertura válida de elevación
import numpy as np

with rasterio.open(ELEVATION_PATH) as src:
    elevation = src.read(1, masked=True)

    valid_mask = ~elevation.mask

print("Pixeles válidos del raster:", valid_mask.sum())
print("Pixeles NoData del raster:", elevation.mask.sum())

Pixeles válidos del raster: 4081208
Pixeles NoData del raster: 5668592


In [ ]:
from rasterio.features import geometry_mask

coverage_results = []

with rasterio.open(ELEVATION_PATH) as src:
    elevation_data = src.read(1, masked=True)

    for _, parcel in parcels_6372.iterrows():

        mask = geometry_mask(
            [parcel.geometry],
            transform=src.transform,
            invert=True,
            out_shape=(src.height, src.width),
        )

        parcel_pixels = mask.sum()

        if parcel_pixels == 0:
            valid_pixels = 0
            valid_percentage = 0.0
        else:
            valid_pixels = (~elevation_data.mask & mask).sum()
            valid_percentage = (
                valid_pixels / parcel_pixels
            ) * 100

        coverage_results.append({
            "ID_POLIGON": parcel["ID_POLIGON"],
            "total_pixels": parcel_pixels,
            "valid_pixels": valid_pixels,
            "valid_percentage": valid_percentage,
        })

coverage_df = gpd.pd.DataFrame(coverage_results)

print(coverage_df.head())
print()
print("Parcelas analizadas:", len(coverage_df))
print(
    "Cobertura válida mínima:",
    coverage_df["valid_percentage"].min()
)
print(
    "Cobertura válida máxima:",
    coverage_df["valid_percentage"].max()
)
print(
    "Cobertura válida promedio:",
    coverage_df["valid_percentage"].mean()
)

  ID_POLIGON  total_pixels  valid_pixels  valid_percentage
0    AGC_196             7             7             100.0
1    AGC_044             1             1             100.0
2    AGC_020             7             7             100.0
3    AGC_174             5             5             100.0
4    AGC_083             2             2             100.0

Parcelas analizadas: 197
Cobertura válida mínima: 0.0
Cobertura válida máxima: 100.0
Cobertura válida promedio: 91.22552574329225


In [ ]:
# Resumen de la cobertura válida por parcela

print("Distribución de cobertura válida (%):")
print(
    coverage_df["valid_percentage"]
    .describe()
)

print("\nParcelas con cobertura menor al 100%:")
print(
    coverage_df[
        coverage_df["valid_percentage"] < 100
    ]
    .sort_values("valid_percentage")
    .to_string(index=False)
)

Distribución de cobertura válida (%):
count    197.000000
mean      91.225526
std       28.180271
min        0.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: valid_percentage, dtype: float64

Parcelas con cobertura menor al 100%:
ID_POLIGON  total_pixels  valid_pixels  valid_percentage
   AGC_009             0             0          0.000000
   AGC_119             0             0          0.000000
   AGC_064             0             0          0.000000
   AGC_067             0             0          0.000000
   AGC_092             0             0          0.000000
   AGC_105             0             0          0.000000
   AGC_135             0             0          0.000000
   AGC_057             0             0          0.000000
   AGC_073             0             0          0.000000
   AGC_032             0             0          0.000000
   AGC_195             0             0          0.000000
   AGC_127             0             0  

In [ ]:
# Comparar el conteo de píxeles usando dos criterios:
# 1. Solo centros de píxel dentro de la parcela
# 2. Cualquier píxel que toque la parcela

comparison = []

with rasterio.open(ELEVATION_PATH) as src:

    for _, parcel in parcels_6372.iterrows():

        mask_center = geometry_mask(
            [parcel.geometry],
            transform=src.transform,
            invert=True,
            out_shape=(src.height, src.width),
            all_touched=False,
        )

        mask_touched = geometry_mask(
            [parcel.geometry],
            transform=src.transform,
            invert=True,
            out_shape=(src.height, src.width),
            all_touched=True,
        )

        comparison.append({
            "ID_POLIGON": parcel["ID_POLIGON"],
            "pixels_center": mask_center.sum(),
            "pixels_touched": mask_touched.sum(),
        })

comparison_df = gpd.pd.DataFrame(comparison)

print(
    comparison_df[
        comparison_df["pixels_center"] == 0
    ].to_string(index=False)
)

print("\nResumen:")
print("Parcelas con 0 píxeles por centro:",
      (comparison_df["pixels_center"] == 0).sum())

print("Parcelas con 0 píxeles tocados:",
      (comparison_df["pixels_touched"] == 0).sum())

ID_POLIGON  pixels_center  pixels_touched
   AGC_009              0               4
   AGC_119              0               3
   AGC_064              0               9
   AGC_067              0               5
   AGC_092              0               2
   AGC_105              0               8
   AGC_135              0               6
   AGC_057              0               4
   AGC_073              0               7
   AGC_032              0               6
   AGC_195              0               6
   AGC_127              0               4
   AGC_017              0               6
   AGC_076              0               5
   AGC_062              0               8
   AGC_139              0               7
   AGC_069              0               2

Resumen:
Parcelas con 0 píxeles por centro: 17
Parcelas con 0 píxeles tocados: 0


In [ ]:
# Revisar el tamaño de las parcelas afectadas
small_parcels = (
    parcels[
        parcels["ID_POLIGON"].isin(
            comparison_df.loc[
                comparison_df["pixels_center"] == 0,
                "ID_POLIGON"
            ]
        )
    ][["ID_POLIGON", "área_ha", "Municipio", "Estado"]]
    .sort_values("área_ha")
)

print(small_parcels.to_string(index=False))

ID_POLIGON  área_ha             Municipio  Estado
   AGC_009     0.38 Cuautepec de Hinojosa Hidalgo
   AGC_069     0.38          Chignahuapan  Puebla
   AGC_092     0.50          Chignahuapan  Puebla
   AGC_057     0.63          Chignahuapan  Puebla
   AGC_119     0.64 Cuautepec de Hinojosa Hidalgo
   AGC_127     0.91          Chignahuapan  Puebla
   AGC_076     1.01          Chignahuapan  Puebla
   AGC_032     1.44          Chignahuapan  Puebla
   AGC_067     1.53           Singuilucan Hidalgo
   AGC_017     1.60          Chignahuapan  Puebla
   AGC_135     1.62          Chignahuapan  Puebla
   AGC_139     1.92          Chignahuapan  Puebla
   AGC_195     1.92          Chignahuapan  Puebla
   AGC_062     2.00          Chignahuapan  Puebla
   AGC_073     2.05          Chignahuapan  Puebla
   AGC_064     2.45 Cuautepec de Hinojosa Hidalgo
   AGC_105     2.99          Chignahuapan  Puebla


In [ ]:
# Relación entre el tamaño de parcela y la resolución del raster

PIXEL_AREA_HA = (120 * 120) / 10_000

diagnostic_df = comparison_df.merge(
    parcels[["ID_POLIGON", "área_ha"]],
    on="ID_POLIGON",
    how="left"
)

diagnostic_df["pixel_area_ha"] = PIXEL_AREA_HA

diagnostic_df["parcel_vs_pixel_pct"] = (
    diagnostic_df["área_ha"] /
    diagnostic_df["pixel_area_ha"]
) * 100

print(
    diagnostic_df[
        [
            "ID_POLIGON",
            "área_ha",
            "pixels_touched",
            "pixel_area_ha",
            "parcel_vs_pixel_pct",
        ]
    ]
    .sort_values("área_ha")
    .head(20)
    .to_string(index=False)
)

ID_POLIGON  área_ha  pixels_touched  pixel_area_ha  parcel_vs_pixel_pct
   AGC_091     0.35               1           1.44            24.305556
   AGC_009     0.38               4           1.44            26.388889
   AGC_069     0.38               2           1.44            26.388889
   AGC_092     0.50               2           1.44            34.722222
   AGC_057     0.63               4           1.44            43.750000
   AGC_119     0.64               3           1.44            44.444444
   AGC_044     0.68               4           1.44            47.222222
   AGC_049     0.87               4           1.44            60.416667
   AGC_127     0.91               4           1.44            63.194444
   AGC_076     1.01               5           1.44            70.138889
   AGC_099     1.17               5           1.44            81.250000
   AGC_157     1.18               4           1.44            81.944444
   AGC_142     1.29               5           1.44            89

In [ ]:
# Estadísticas preliminares de elevación por parcela
# usando los píxeles que tocan cada geometría.
# USANDO all_touched=True
#Tomamos los píxeles que tocan la parcela y calculamos estadísticas. 
# Ventaja: ninguna parcela se queda sin información
#Problema: un píxel que apenas toca una esquina pesa igual que uno que cubre una parte mucho mayor.

elevation_stats = []

with rasterio.open(ELEVATION_PATH) as src:
    elevation_data = src.read(1, masked=True)

    for _, parcel in parcels_6372.iterrows():

        mask = geometry_mask(
            [parcel.geometry],
            transform=src.transform,
            invert=True,
            out_shape=(src.height, src.width),
            all_touched=True,
        )

        values = elevation_data[mask]

        # Eliminar NoData
        values = values.compressed()

        if len(values) == 0:
            stats = {
                "elev_min": np.nan,
                "elev_max": np.nan,
                "elev_mean": np.nan,
                "elev_std": np.nan,
                "valid_pixels": 0,
            }
        else:
            stats = {
                "elev_min": values.min(),
                "elev_max": values.max(),
                "elev_mean": values.mean(),
                "elev_std": values.std(),
                "valid_pixels": len(values),
            }

        elevation_stats.append({
            "ID_POLIGON": parcel["ID_POLIGON"],
            **stats
        })

elevation_stats_df = gpd.pd.DataFrame(elevation_stats)

print(elevation_stats_df.head(10).to_string(index=False))

print("\nParcelas analizadas:", len(elevation_stats_df))
print(
    "Parcelas sin elevación válida:",
    elevation_stats_df["elev_mean"].isna().sum()
)

ID_POLIGON  elev_min  elev_max   elev_mean  elev_std  valid_pixels
   AGC_196    2539.0    2550.0 2544.769287  3.576716            13
   AGC_044    2529.0    2531.0 2530.000000  0.707107             4
   AGC_020    2528.0    2531.0 2529.090820  0.995859            11
   AGC_174    2542.0    2553.0 2546.545410  3.652238            11
   AGC_083    2557.0    2569.0 2563.250000  4.115520             8
   AGC_149    2631.0    2672.0 2649.500000 10.430990            18
   AGC_052    2616.0    2647.0 2630.818115  9.271836            11
   AGC_009    2630.0    2661.0 2646.750000 11.388042             4
   AGC_119    2630.0    2652.0 2644.333252 10.143415             3
   AGC_064    2645.0    2670.0 2655.555664  7.689981             9

Parcelas analizadas: 197
Parcelas sin elevación válida: 0


In [ ]:
elevation_stats_df["elev_range"] = (
    elevation_stats_df["elev_max"]
    - elevation_stats_df["elev_min"]
)

print("Resumen de elevación media:")
print(elevation_stats_df["elev_mean"].describe())

print("\nResumen de variabilidad interna:")
print(elevation_stats_df["elev_std"].describe())

print("\nResumen del rango de elevación:")
print(elevation_stats_df["elev_range"].describe())

Resumen de elevación media:
count     197.000000
mean     2667.135742
std        92.172882
min      2497.199951
25%      2611.500000
50%      2638.545410
75%      2726.000000
max      2905.250000
Name: elev_mean, dtype: float64

Resumen de variabilidad interna:
count    197.000000
mean       5.349356
std        3.942338
min        0.000000
25%        2.285218
50%        4.166458
75%        7.780952
max       17.503124
Name: elev_std, dtype: float64

Resumen del rango de elevación:
count    197.000000
mean      17.142132
std       13.471439
min        0.000000
25%        7.000000
50%       13.000000
75%       24.000000
max       58.000000
Name: elev_range, dtype: float64


In [ ]:
SLOPE_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "topography"
    / "Pendiente_INEGI_CEM4_120m_grados.tif"
)

with rasterio.open(ELEVATION_PATH) as elev_src:
    with rasterio.open(SLOPE_PATH) as slope_src:

        print("ELEVACIÓN")
        print("CRS:", elev_src.crs)
        print("Resolución:", elev_src.res)
        print("Shape:", (elev_src.height, elev_src.width))
        print("Bounds:", elev_src.bounds)

        print("\nPENDIENTE")
        print("CRS:", slope_src.crs)
        print("Resolución:", slope_src.res)
        print("Shape:", (slope_src.height, slope_src.width))
        print("Bounds:", slope_src.bounds)
        print("NoData:", slope_src.nodata)

ELEVACIÓN
CRS: EPSG:6372
Resolución: (120.0, 120.0)
Shape: (3362, 2900)
Bounds: BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)

PENDIENTE
CRS: EPSG:6372
Resolución: (120.0, 120.0)
Shape: (3362, 2900)
Bounds: BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)
NoData: -9999.0


In [ ]:
slope_stats = []

with rasterio.open(SLOPE_PATH) as src:
    slope_data = src.read(1, masked=True)

    for _, parcel in parcels_6372.iterrows():

        mask = geometry_mask(
            [parcel.geometry],
            transform=src.transform,
            invert=True,
            out_shape=(src.height, src.width),
            all_touched=True,
        )

        values = slope_data[mask]
        values = values.compressed()

        if len(values) == 0:
            stats = {
                "slope_min": np.nan,
                "slope_max": np.nan,
                "slope_mean": np.nan,
                "slope_std": np.nan,
                "valid_pixels": 0,
            }
        else:
            stats = {
                "slope_min": values.min(),
                "slope_max": values.max(),
                "slope_mean": values.mean(),
                "slope_std": values.std(),
                "valid_pixels": len(values),
            }

        slope_stats.append({
            "ID_POLIGON": parcel["ID_POLIGON"],
            **stats
        })

slope_stats_df = gpd.pd.DataFrame(slope_stats)

print(slope_stats_df.head(10).to_string(index=False))

print("\nParcelas analizadas:", len(slope_stats_df))
print(
    "Parcelas sin pendiente válida:",
    slope_stats_df["slope_mean"].isna().sum()
)

ID_POLIGON  slope_min  slope_max  slope_mean  slope_std  valid_pixels
   AGC_196   0.778125   2.271474    1.632749   0.428928            13
   AGC_044   0.304322   0.566185    0.430841   0.107087             3
   AGC_020   0.304322   0.778125    0.511222   0.164792             8
   AGC_174   0.680460   5.762671    2.582436   1.246397            11
   AGC_083   0.932198   2.904267    2.117744   0.607298             8
   AGC_149   0.513400  14.943816    7.468832   4.145267            18
   AGC_052   2.294827  13.417810    6.797590   3.118593            11
   AGC_009   6.546266   8.264517    7.295655   0.732785             4
   AGC_119   6.627557   8.264517    7.584344   0.696322             3
   AGC_064   2.868600   8.264517    5.805400   1.364113             9

Parcelas analizadas: 197
Parcelas sin pendiente válida: 0


In [ ]:
print("Resumen de pendiente media:")
print(slope_stats_df["slope_mean"].describe())

print("\nResumen de variabilidad interna:")
print(slope_stats_df["slope_std"].describe())

slope_stats_df["slope_range"] = (
    slope_stats_df["slope_max"]
    - slope_stats_df["slope_min"]
)

print("\nResumen del rango de pendiente:")
print(slope_stats_df["slope_range"].describe())

Resumen de pendiente media:
count    197.000000
mean       2.875671
std        2.090660
min        0.282649
25%        1.278814
50%        2.285353
75%        4.109772
max        8.930869
Name: slope_mean, dtype: float64

Resumen de variabilidad interna:
count    197.000000
mean       0.865657
std        0.914511
min        0.000000
25%        0.260248
50%        0.525752
75%        1.116439
max        4.810753
Name: slope_std, dtype: float64

Resumen del rango de pendiente:
count    197.000000
mean       2.793045
std        3.134011
min        0.000000
25%        0.787261
50%        1.752022
75%        3.619881
max       15.418815
Name: slope_range, dtype: float64
